# trainer-class-skeleton — faded example 1: Trainer skeleton with step counter increment

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `trainer-class-skeleton`. Running the beacon reports progress on the `Trainer: Trainer class skeleton` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Trainer: Trainer class skeleton` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`trainer-class-skeleton`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "trainer-class-skeleton"
DD_SUBTOPIC = "Trainer: Trainer class skeleton"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

In the canonical Trainer pattern, a global step counter `self.step` is incremented once per batch (not once per epoch). It starts at 0 in `__init__` and goes up by 1 each time `optimizer.step()` is called. After `n_epochs` epochs with `B` batches per epoch, `self.step` equals `n_epochs * B`.

## Faded exercise 1

Implement `FadedTrainer1` with the canonical skeleton: `__init__` stores model, optimizer, train_loader, val_loader, loss_fn; initializes `self.step = 0` and `self.history`. `_step(x, y)` returns `loss_fn(model(x), y)`. `fit(n_epochs)` runs the full loop including backward/step/zero_grad and calls `validate()` each epoch. `validate()` computes weighted-average val loss under `t.inference_mode()`. The only blank is the step counter increment.

**Fill in:** The self.step += 1 statement that increments the global step counter after each optimizer step.

In [ ]:
import torch as t
import torch.nn as nn

class FadedTrainer1:
    def __init__(self, model, optimizer, train_loader, val_loader, loss_fn):
        self.model = model
        self.optimizer = optimizer
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.loss_fn = loss_fn
        self.step = 0
        self.history = {'train_loss': [], 'val_loss': []}

    def _step(self, x, y):
        return self.loss_fn(self.model(x), y)

    def fit(self, n_epochs):
        for _ in range(n_epochs):
            self.model.train()
            for x, y in self.train_loader:
                loss = self._step(x, y)
                loss.backward()
                self.optimizer.step()
                self.optimizer.zero_grad()
                self.step += 1
                self.history['train_loss'].append(loss.item())
            self.validate()

    def validate(self):
        self.model.eval()
        total, count = 0.0, 0
        with t.inference_mode():
            for x, y in self.val_loader:
                loss = self.loss_fn(self.model(x), y)
                total += loss.item() * x.shape[0]
                count += x.shape[0]
        self.history['val_loss'].append(total / count)


def _test():
    import torch as t
    import torch.nn as nn
    from torch.utils.data import DataLoader, TensorDataset
    t.manual_seed(0)
    X = t.randn(20, 3)
    Y = X[:, 0:1]
    train_dl = DataLoader(TensorDataset(X[:16], Y[:16]), batch_size=4)
    val_dl = DataLoader(TensorDataset(X[16:], Y[16:]), batch_size=4)
    model = nn.Linear(3, 1)
    opt = t.optim.SGD(model.parameters(), lr=0.01)
    trainer = FadedTrainer1(model, opt, train_dl, val_dl, nn.MSELoss())
    trainer.fit(2)
    # 2 epochs * 4 batches = 8 steps
    assert trainer.step == 8, f'expected 8 steps, got {trainer.step}'
    assert len(trainer.history['val_loss']) == 2


try:
    _test()
    _dd_passed.add('faded1')
    print('[Delta Drills] faded1 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import torch as t
import torch.nn as nn

class FadedTrainer1:
    def __init__(self, model, optimizer, train_loader, val_loader, loss_fn):
        self.model = model
        self.optimizer = optimizer
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.loss_fn = loss_fn
        self.step = 0
        self.history = {'train_loss': [], 'val_loss': []}

    def _step(self, x, y):
        return self.loss_fn(self.model(x), y)

    def fit(self, n_epochs):
        for _ in range(n_epochs):
            self.model.train()
            for x, y in self.train_loader:
                loss = self._step(x, y)
                loss.backward()
                self.optimizer.step()
                self.optimizer.zero_grad()
                self.step += 1
                self.history['train_loss'].append(loss.item())
            self.validate()

    def validate(self):
        self.model.eval()
        total, count = 0.0, 0
        with t.inference_mode():
            for x, y in self.val_loader:
                loss = self.loss_fn(self.model(x), y)
                total += loss.item() * x.shape[0]
                count += x.shape[0]
        self.history['val_loss'].append(total / count)
```
</details>